In [ ]:

import pandas as pd

# Load the train and eval datasets
train_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\ml_benchmark\\08_santander_value\\split_train.csv'
eval_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark\\ml_benchmark\\08_santander_value\\split_eval.csv'

train_df = pd.read_csv(train_path)
eval_df = pd.read_csv(eval_path)

# Display the first few rows of the train and eval datasets
print("Train Data:")
print(train_df.head())
print("\nEval Data:")
print(eval_df.head())

# Display basic statistics of the train dataset
print("\nTrain Data Statistics:")
print(train_df.describe())

# Display information about the train dataset
print("\nTrain Data Info:")
print(train_df.info())

# Display basic statistics of the eval dataset
print("\nEval Data Statistics:")
print(eval_df.describe())

# Display information about the eval dataset
print("\nEval Data Info:")
print(eval_df.info())


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

00000e+07  ...  3.600000e+07  1.000000e+08

[8 rows x 19 columns]

Train Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3567 entries, 0 to 3566
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   target      3567 non-null   float64
 1   48df886f9   3567 non-null   float64
 2   0deb4b6a8   3567 non-null   int64  
 3   34b15f335   3567 non-null   float64
 4   a8cb14b00   3567 non-null   int64  
 5   2f0771a37   3567 non-null   int64  
 6   s30347e683  3567 non-null   int64  
 7   d08d1fbe3   3567 non-null   int64  
 8   6ee66e115   3567 non-null   int64  
 9   20aa07010   3567 non-null   float64
 10  dc5a8f1d8   3567 non-null   float64
 11  11d86fa6a   3567 non-null   float64
 12  77c9823f2   3567 non-null   int64  
 13  8d6c2a0b2   3567 non-nu

In [ ]:


import numpy as np

# Check for outliers in the target variable
print("Target Variable Outliers in Train Data:")
print(train_df['target'].describe(percentiles=[0.01, 0.05, 0.95, 0.99]))

# Log-transform the target variable to handle the wide range of values
train_df['target_log'] = np.log1p(train_df['target'])
eval_df['target_log'] = np.log1p(eval_df['target'])

# Display the new target variable after log transformation
print("\nLog-Transformed Target Variable in Train Data:")
print(train_df['target_log'].describe())

# Drop the original target column
train_df.drop(columns=['target'], inplace=True)
eval_df.drop(columns=['target'], inplace=True)

# Display the first few rows of the train and eval datasets after preprocessing
print("\nTrain Data After Preprocessing:")
print(train_df.head())
print("\nEval Data After Preprocessing:")
print(eval_df.head())



Target Variable Outliers in Train Data:
count    3.567000e+03
mean     5.964817e+06
std      8.229457e+06
min      3.000000e+04
1%       3.500000e+04
5%       6.830000e+04
50%      2.305111e+06
95%      2.400000e+07
99%      3.847809e+07
max      4.000000e+07
Name: target, dtype: float64

Log-Transformed Target Variable in Train Data:
count    3567.000000
mean       14.483176
std         1.766069
min        10.308986
25%        13.304687
50%        14.650640
75%        15.894952
max        17.504390
Name: target_log, dtype: float64

Train Data After Preprocessing:
   48df886f9  0deb4b6a8  34b15f335  ...  96f83a237  b8a716ebf  target_log
0        0.0          0        0.0  ...          0        0.0   16.182634
1        0.0          0        0.0  ...          0        0.0   15.201805
2        0.0          0        0.0  ...          0        0.0   13.710151
3        0.0          0        0.0  ...          0        0.0   14.509158
4        0.0          0        0.0  ...          0        0

In [ ]:


from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_log_error
import numpy as np

# Define the features and the target variable
X_train = train_df.drop(columns=['target_log'])
y_train = train_df['target_log']
X_eval = eval_df.drop(columns=['target_log'])
y_eval = eval_df['target_log']

# Train a Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Make predictions on the eval data
y_pred_log = model.predict(X_eval)

# Convert the predictions back to the original scale
y_pred = np.expm1(y_pred_log)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_eval_original, y_pred))

# Print the RMSLE
print(f"RMSLE on the eval data: {rmsle:.4f}")


RMSLE on the eval data: 1.6872


In [ ]:



import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
import numpy as np

# Convert the data to DMatrix format for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
deval = xgb.DMatrix(X_eval, label=y_eval)

# Define the XGBoost parameters
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'eta': 0.1,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'seed': 42
}

# Train the XGBoost model
num_rounds = 100
bst = xgb.train(params, dtrain, num_rounds)

# Make predictions on the eval data
y_pred_log = bst.predict(deval)

# Convert the predictions back to the original scale
y_pred = np.expm1(y_pred_log)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_eval_original, y_pred))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost): {rmsle:.4f}")



RMSLE on the eval data (XGBoost): 1.6234


In [ ]:



from sklearn.model_selection import GridSearchCV

# Define the parameter grid for grid search
param_grid = {
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

# Convert the data to DMatrix format for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
deval = xgb.DMatrix(X_eval, label=y_eval)

# Define the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse', seed=42)

# Perform grid search
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='neg_mean_squared_log_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best parameters
best_params = grid_search.best_params_
print(f"Best parameters: {best_params}")

# Train the model with the best parameters
best_model = xgb.XGBRegressor(**best_params, objective='reg:squarederror', eval_metric='rmse', seed=42)
best_model.fit(X_train, y_train)

# Make predictions on the eval data
y_pred_log = best_model.predict(X_eval)

# Convert the predictions back to the original scale
y_pred = np.expm1(y_pred_log)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_eval_original, y_pred))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost with tuned hyperparameters): {rmsle:.4f}")



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

`__sklearn_tags__` will raise an error.
  warnings.warn(
---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
Cell In[6], line 30
     28 grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, cv=3, scoring='neg_mean_squared_log_error', n_jobs=-1)
     29 print('##active_line20##')
---> 30 grid_search.fit(X_train, y_train)
     31 print('##active_line21##')
     32 pass

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\base.py:1389, in _fit_context.<locals>.decorator.<locals>.wrapper(estimator, *args, **kwargs)
   1382     estimator._validate_params()
   1384 with config_context(
   1385     skip_parameter_validation=(
   1386         prefer_skip_nested_validation

In [ ]:



from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# Define the parameter distribution for randomized search
param_dist = {
    'max_depth': randint(3, 10),
    'learning_rate': uniform(0.01, 0.2),
    'subsample': uniform(0.7, 0.3),
    'colsample_bytree': uniform(0.7, 0.3)
}

# Convert the data to DMatrix format for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
deval = xgb.DMatrix(X_eval, label=y_eval)

# Define the XGBoost model
xgb_model = xgb.XGBRegressor(objective='reg:squarederror', eval_metric='rmse', seed=42)

# Perform randomized search
random_search = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_dist, n_iter=10, cv=3, scoring='neg_mean_squared_log_error', n_jobs=-1, random_state=42)
random_search.fit(X_train, y_train)

# Get the best parameters
best_params = random_search.best_params_
print(f"Best parameters: {best_params}")

# Train the model with the best parameters
best_model = xgb.XGBRegressor(**best_params, objective='reg:squarederror', eval_metric='rmse', seed=42)
best_model.fit(X_train, y_train)

# Make predictions on the eval data
y_pred_log = best_model.predict(X_eval)

# Convert the predictions back to the original scale
y_pred = np.expm1(y_pred_log)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_eval_original, y_pred))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost with tuned hyperparameters): {rmsle:.4f}")



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

ngs.warn(
---------------------------------------------------------------------------
AttributeError                            Traceback (most recent call last)
Cell In[7], line 32
     30 random_search = RandomizedSearchCV(estimator=xgb_model, param_distributions=param_dist, n_iter=10, cv=3, scoring='neg_mean_squared_log_error', n_jobs=-1, random_state=42)
     31 print('##active_line21##')
---> 32 random_search.fit(X_train, y_train)
     33 print('##active_line22##')
     34 pass

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\base.py:1389, in _fit_context.<locals>.decorator.<locals>.wrapper(estimator, *args, **kwargs)
   1382     estimator._validate_params()
   1384 with config_context(
   1385     skip_parameter_validation=(
   1386         prefer_skip_nested_validation

In [ ]:



import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
import numpy as np

# Define the parameter grid for hyperparameter tuning
param_grid = {
    'max_depth': [4, 6, 8],
    'eta': [0.01, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

# Convert the data to DMatrix format for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
deval = xgb.DMatrix(X_eval, label=y_eval)

# Define the base parameters
base_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'seed': 42
}

# Perform grid search manually
best_params = None
best_rmsle = float('inf')

for max_depth in param_grid['max_depth']:
    for eta in param_grid['eta']:
        for subsample in param_grid['subsample']:
            for colsample_bytree in param_grid['colsample_bytree']:
                params = base_params.copy()
                params.update({
                    'max_depth': max_depth,
                    'eta': eta,
                    'subsample': subsample,
                    'colsample_bytree': colsample_bytree
                })

                # Perform cross-validation
                cv_results = xgb.cv(params, dtrain, num_boost_round=100, nfold=3, metrics='rmse', early_stopping_rounds=10, seed=42)
                final_rmse = cv_results['test-rmse-mean'].iloc[-1]
                final_rmsle = np.sqrt(final_rmse)

                if final_rmsle < best_rmsle:
                    best_rmsle = final_rmsle
                    best_params = params

print(f"Best parameters: {best_params}")
print(f"Best RMSLE from cross-validation: {best_rmsle:.4f}")

# Train the model with the best parameters
num_rounds = 100
bst = xgb.train(best_params, dtrain, num_rounds)

# Make predictions on the eval data
y_pred_log = bst.predict(deval)

# Convert the predictions back to the original scale
y_pred = np.expm1(y_pred_log)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_eval_original, y_pred))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost with tuned hyperparameters): {rmsle:.4f}")



Best parameters: {'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'seed': 42, 'max_depth': 6, 'eta': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.8}
Best RMSLE from cross-validation: 1.2885
RMSLE on the eval data (XGBoost with tuned hyperparameters): 1.6234


In [ ]:



# Feature Engineering
# Create interaction terms
X_train['feature1_feature2'] = X_train['48df886f9'] * X_train['0deb4b6a8']
X_eval['feature1_feature2'] = X_eval['48df886f9'] * X_eval['0deb4b6a8']

# Create polynomial features
X_train['feature1_squared'] = X_train['48df886f9'] ** 2
X_eval['feature1_squared'] = X_eval['48df886f9'] ** 2

# Convert the data to DMatrix format for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
deval = xgb.DMatrix(X_eval, label=y_eval)

# Train the model with the best parameters
num_rounds = 100
bst = xgb.train(best_params, dtrain, num_rounds)

# Make predictions on the eval data
y_pred_log = bst.predict(deval)

# Convert the predictions back to the original scale
y_pred = np.expm1(y_pred_log)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_eval_original, y_pred))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost with feature engineering): {rmsle:.4f}")




RMSLE on the eval data (XGBoost with feature engineering): 1.6196


In [ ]:




# Feature Selection
# Use feature importance from XGBoost to select the most important features
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': bst.get_score(importance_type='weight').values()
}).sort_values(by='importance', ascending=False)

# Print the feature importances
print("Feature Importances:")
print(importance_df)

# Select the top k features
k = 10
top_features = importance_df['feature'].head(k).tolist()

# Filter the datasets to keep only the top features
X_train_selected = X_train[top_features]
X_eval_selected = X_eval[top_features]

# Convert the data to DMatrix format for XGBoost
dtrain_selected = xgb.DMatrix(X_train_selected, label=y_train)
deval_selected = xgb.DMatrix(X_eval_selected, label=y_eval)

# Train the model with the selected features
num_rounds = 100
bst_selected = xgb.train(best_params, dtrain_selected, num_rounds)

# Make predictions on the eval data
y_pred_log_selected = bst_selected.predict(deval_selected)

# Convert the predictions back to the original scale
y_pred_selected = np.expm1(y_pred_log_selected)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle_selected = np.sqrt(mean_squared_log_error(y_eval_original, y_pred_selected))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost with feature selection): {rmsle_selected:.4f}")



---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[10], line 6
      4 pass
      5 print('##active_line3##')
----> 6 importance_df = pd.DataFrame({'feature': X_train.columns, 'importance': bst.get_score(importance_type='weight').values()}).sort_values(by='importance', ascending=False)
      7 print('##active_line7##')
      8 pass

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\frame.py:778, in DataFrame.__init__(self, data, index, columns, dtype, copy)
    772     mgr = self._init_mgr(
    773         data, axes={"index": index, "columns": columns}, dtype=dtype, copy=copy
    774     )
    776 elif isinstance(data, dict):
    777     # GH#38939 de facto copy defaults to False only in non-dict cases
--> 778     mgr = dict_to_mgr(data, index, columns, dtype=dtype, copy=copy, typ=manager)
    779 elif isinstance(data, ma.Masked

In [ ]:




# Debugging: Print the lengths and contents of the feature names and importance scores
feature_names = X_train.columns
importance_scores = bst.get_score(importance_type='weight')

print(f"Number of feature names: {len(feature_names)}")
print(f"Number of importance scores: {len(importance_scores)}")

print("Feature names:")
print(feature_names)

print("Importance scores:")
print(importance_scores)


Number of feature names: 20
Number of importance scores: 19
Feature names:
Index(['48df886f9', '0deb4b6a8', '34b15f335', 'a8cb14b00', '2f0771a37',
       's30347e683', 'd08d1fbe3', '6ee66e115', '20aa07010', 'dc5a8f1d8',
       '11d86fa6a', '77c9823f2', '8d6c2a0b2', '4681de4fd', 'adf119b9a',
       'cff75dd09', '96f83a237', 'b8a716ebf', 'feature1_feature2',
       'feature1_squared'],
      dtype='object')
Importance scores:
{'48df886f9': 82.0, '0deb4b6a8': 27.0, '34b15f335': 53.0, 'a8cb14b00': 3.0, '2f0771a37': 12.0, 's30347e683': 65.0, 'd08d1fbe3': 33.0, '6ee66e115': 13.0, '20aa07010': 867.0, 'dc5a8f1d8': 183.0, '11d86fa6a': 206.0, '77c9823f2': 4.0, '8d6c2a0b2': 15.0, '4681de4fd': 483.0, 'adf119b9a': 157.0, 'cff75dd09': 78.0, '96f83a237': 50.0, 'b8a716ebf': 319.0, 'feature1_squared': 13.0}


In [ ]:




# Re-train the model with feature engineering included
# Create interaction terms
X_train['feature1_feature2'] = X_train['48df886f9'] * X_train['0deb4b6a8']
X_eval['feature1_feature2'] = X_eval['48df886f9'] * X_eval['0deb4b6a8']

# Create polynomial features
X_train['feature1_squared'] = X_train['48df886f9'] ** 2
X_eval['feature1_squared'] = X_eval['48df886f9'] ** 2

# Convert the data to DMatrix format for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
deval = xgb.DMatrix(X_eval, label=y_eval)

# Train the model with the best parameters
num_rounds = 100
bst = xgb.train(best_params, dtrain, num_rounds)

# Extract feature importances
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': [bst.get_score(importance_type='weight').get(f, 0) for f in X_train.columns]
}).sort_values(by='importance', ascending=False)

# Print the feature importances
print("Feature Importances:")
print(importance_df)

# Select the top k features
k = 10
top_features = importance_df['feature'].head(k).tolist()

# Filter the datasets to keep only the top features
X_train_selected = X_train[top_features]
X_eval_selected = X_eval[top_features]

# Convert the data to DMatrix format for XGBoost
dtrain_selected = xgb.DMatrix(X_train_selected, label=y_train)
deval_selected = xgb.DMatrix(X_eval_selected, label=y_eval)

# Train the model with the selected features
num_rounds = 100
bst_selected = xgb.train(best_params, dtrain_selected, num_rounds)

# Make predictions on the eval data
y_pred_log_selected = bst_selected.predict(deval_selected)

# Convert the predictions back to the original scale
y_pred_selected = np.expm1(y_pred_log_selected)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle_selected = np.sqrt(mean_squared_log_error(y_eval_original, y_pred_selected))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost with feature selection): {rmsle_selected:.4f}")




Feature Importances:
              feature  importance
8           20aa07010       867.0
13          4681de4fd       483.0
17          b8a716ebf       319.0
10          11d86fa6a       206.0
9           dc5a8f1d8       183.0
14          adf119b9a       157.0
0           48df886f9        82.0
15          cff75dd09        78.0
5          s30347e683        65.0
2           34b15f335        53.0
16          96f83a237        50.0
6           d08d1fbe3        33.0
1           0deb4b6a8        27.0
12          8d6c2a0b2        15.0
19   feature1_squared        13.0
7           6ee66e115        13.0
4           2f0771a37        12.0
11          77c9823f2         4.0
3           a8cb14b00         3.0
18  feature1_feature2         0.0
RMSLE on the eval data (XGBoost with feature selection): 1.6358


In [ ]:





# Further Feature Engineering
# Create more interaction terms
X_train['feature1_feature3'] = X_train['48df886f9'] * X_train['34b15f335']
X_eval['feature1_feature3'] = X_eval['48df886f9'] * X_eval['34b15f335']

# Create ratio features
X_train['feature1_ratio'] = X_train['48df886f9'] / X_train['34b15f335']
X_eval['feature1_ratio'] = X_eval['48df886f9'] / X_eval['34b15f335']

# Convert the data to DMatrix format for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train)
deval = xgb.DMatrix(X_eval, label=y_eval)

# Train the model with the best parameters
num_rounds = 100
bst = xgb.train(best_params, dtrain, num_rounds)

# Make predictions on the eval data
y_pred_log = bst.predict(deval)

# Convert the predictions back to the original scale
y_pred = np.expm1(y_pred_log)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_eval_original, y_pred))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost with additional feature engineering): {rmsle:.4f}")

# Extract feature importances
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': [bst.get_score(importance_type='weight').get(f, 0) for f in X_train.columns]
}).sort_values(by='importance', ascending=False)

# Print the feature importances
print("Feature Importances:")
print(importance_df)



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

eight, base_margin, missing, silent, feature_names, feature_types, nthread, group, qid, label_lower_bound, label_upper_bound, feature_weights, enable_categorical, data_split_mode)
    875     assert self.handle is not None
    876     return
--> 878 handle, feature_names, feature_types = dispatch_data_backend(
    879     data,
    880     missing=self.missing,
    881     threads=self.nthread,
    882     feature_names=feature_names,
    883     feature_types=feature_types,
    884     enable_categorical=enable_categorical,
    885     data_split_mode=data_split_mode,
    886 )
    887 assert handle is not None
    888 self.handle = handle

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\xgboost\data.py:1195, in dispatch_data_backend(data, missing, threads, feature_names, feature_ty

In [ ]:




# Handle infinity values
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_eval = X_eval.replace([np.inf, -np.inf], np.nan)

# Convert the data to DMatrix format for XGBoost
dtrain = xgb.DMatrix(X_train, label=y_train, missing=np.nan)
deval = xgb.DMatrix(X_eval, label=y_eval, missing=np.nan)

# Train the model with the best parameters
num_rounds = 100
bst = xgb.train(best_params, dtrain, num_rounds)

# Make predictions on the eval data
y_pred_log = bst.predict(deval)

# Convert the predictions back to the original scale
y_pred = np.expm1(y_pred_log)
y_eval_original = np.expm1(y_eval)

# Calculate the RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_eval_original, y_pred))

# Print the RMSLE
print(f"RMSLE on the eval data (XGBoost with infinity values handled): {rmsle:.4f}")

# Extract feature importances
importance_df = pd.DataFrame({
    'feature': X_train.columns,
    'importance': [bst.get_score(importance_type='weight').get(f, 0) for f in X_train.columns]
}).sort_values(by='importance', ascending=False)

# Print the feature importances
print("Feature Importances:")
print(importance_df)




RMSLE on the eval data (XGBoost with infinity values handled): 1.6241
Feature Importances:
              feature  importance
8           20aa07010       871.0
13          4681de4fd       548.0
17          b8a716ebf       325.0
10          11d86fa6a       199.0
9           dc5a8f1d8       188.0
14          adf119b9a       172.0
15          cff75dd09        84.0
0           48df886f9        77.0
2           34b15f335        74.0
5          s30347e683        57.0
16          96f83a237        42.0
6           d08d1fbe3        34.0
1           0deb4b6a8        19.0
12          8d6c2a0b2        15.0
19   feature1_squared        13.0
4           2f0771a37        12.0
7           6ee66e115        12.0
11          77c9823f2         7.0
21     feature1_ratio         5.0
3           a8cb14b00         2.0
20  feature1_feature3         1.0
18  feature1_feature2         0.0


In [ ]:





# K-Fold Cross-Validation
from sklearn.model_selection import KFold

# Define the number of folds
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Initialize a list to store the RMSLE scores
rmsle_scores = []

# Perform k-fold cross-validation
for train_index, val_index in kf.split(X_train):
    # Split the data
    X_train_kf, X_val_kf = X_train.iloc[train_index], X_train.iloc[val_index]
    y_train_kf, y_val_kf = y_train.iloc[train_index], y_train.iloc[val_index]
    
    # Convert the data to DMatrix format for XGBoost
    dtrain_kf = xgb.DMatrix(X_train_kf, label=y_train_kf, missing=np.nan)
    dval_kf = xgb.DMatrix(X_val_kf, label=y_val_kf, missing=np.nan)
    
    # Train the model
    bst_kf = xgb.train(best_params, dtrain_kf, num_rounds)
    
    # Make predictions on the validation data
    y_pred_log_kf = bst_kf.predict(dval_kf)
    
    # Convert the predictions back to the original scale
    y_pred_kf = np.expm1(y_pred_log_kf)
    y_val_original_kf = np.expm1(y_val_kf)
    
    # Calculate the RMSLE
    rmsle_kf = np.sqrt(mean_squared_log_error(y_val_original_kf, y_pred_kf))
    
    # Append the RMSLE score to the list
    rmsle_scores.append(rmsle_kf)

# Print the RMSLE scores for each fold
print("RMSLE scores for each fold:", rmsle_scores)

# Calculate the mean and standard deviation of the RMSLE scores
mean_rmsle = np.mean(rmsle_scores)
std_rmsle = np.std(rmsle_scores)

# Print the mean and standard deviation of the RMSLE scores
print(f"Mean RMSLE: {mean_rmsle:.4f}")
print(f"Standard Deviation of RMSLE: {std_rmsle:.4f}")



RMSLE scores for each fold: [np.float64(1.64725194243057), np.float64(1.663812615726961), np.float64(1.6969423041641067), np.float64(1.6558122571160472), np.float64(1.7003413579257194)]
Mean RMSLE: 1.6728
Standard Deviation of RMSLE: 0.0217
